In [ ]:
!pip install -q --upgrade --force-reinstall \
    "tensorflow==2.19.0" \
    "tf-keras==2.19.0" \
    "tensorflow-decision-forests==1.12.0" \
    "tensorflowjs==4.22.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 967.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.1/525.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/

In [ ]:
import os

# Le indica a TensorFlow que utilice la implementación
# compatible con Keras 2 para la conversión.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

from google.colab import drive
drive.mount("/content/drive")

import json
import shutil
from pathlib import Path

import numpy as np
import tensorflow as tf
import tf_keras as keras
import tensorflow_decision_forests as tfdf
import tensorflowjs as tfjs

print("TensorFlow:", tf.__version__)
print("TF-Keras:", keras.__version__)
print("TensorFlow Decision Forests:", tfdf.__version__)
print("TensorFlow.js importado correctamente.")

Mounted at /content/drive


VersionError: Detected incompatible Protobuf Gencode/Runtime versions when loading yggdrasil_decision_forests/dataset/data_spec.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.

In [ ]:
import os
import sys
import types
import importlib.metadata as metadata

# Debe establecerse antes de importar TensorFlow.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

from google.colab import drive

# Si Drive ya estaba montado, no lo vuelve a montar.
drive.mount("/content/drive", force_remount=False)

import json
import shutil
from pathlib import Path

import numpy as np
import tensorflow as tf
import tf_keras as keras

# El intento anterior pudo dejar módulos cargados parcialmente.
# Los retiramos antes de crear los reemplazos temporales.
for nombre_modulo in list(sys.modules):
    if (
        nombre_modulo == "tensorflow_decision_forests"
        or nombre_modulo.startswith("tensorflow_decision_forests.")
        or nombre_modulo == "yggdrasil_decision_forests"
        or nombre_modulo.startswith("yggdrasil_decision_forests.")
        or nombre_modulo == "tensorflow_hub"
        or nombre_modulo.startswith("tensorflow_hub.")
    ):
        del sys.modules[nombre_modulo]

# TensorFlow.js importa estas dependencias aunque nuestra conversión
# de un modelo Keras normal no las utiliza.
tfdf_temporal = types.ModuleType(
    "tensorflow_decision_forests"
)
tfdf_temporal.__version__ = "omitido: no requerido"

hub_temporal = types.ModuleType(
    "tensorflow_hub"
)
hub_temporal.__version__ = "omitido: no requerido"

sys.modules[
    "tensorflow_decision_forests"
] = tfdf_temporal

sys.modules[
    "tensorflow_hub"
] = hub_temporal

# Ahora el conversor puede importarse sin cargar TF-DF ni TF-Hub.
import tensorflowjs as tfjs

print("VERSIONES DEL ENTORNO")
print("-" * 45)
print("TensorFlow:", tf.__version__)
print("TF-Keras:", keras.__version__)
print(
    "TensorFlow.js:",
    metadata.version("tensorflowjs")
)
print(
    "TF Decision Forests instalado:",
    metadata.version(
        "tensorflow-decision-forests"
    )
)
print(
    "Protobuf instalado:",
    metadata.version("protobuf")
)

print()
print(
    "TensorFlow.js importado correctamente."
)
print(
    "Conversor Keras disponible:",
    hasattr(
        tfjs.converters,
        "save_keras_model"
    )
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
VERSIONES DEL ENTORNO
---------------------------------------------
TensorFlow: 2.19.0
TF-Keras: 2.19.0
TensorFlow.js: 4.22.0
TF Decision Forests instalado: 1.12.0
Protobuf instalado: 5.29.6

TensorFlow.js importado correctamente.
Conversor Keras disponible: True


In [ ]:
print(
    "Función de exportación:",
    tfjs.converters.save_keras_model
)

Función de exportación: <function save_keras_model at 0x7941e7f256c0>


In [ ]:
BASE_DIR = Path(
    "/content/drive/MyDrive/Taller3_Sign_Language"
)

MODELS_DIR = BASE_DIR / "models"

ruta_pesos_npz = (
    MODELS_DIR / "modelo_final_inferencia_pesos.npz"
)

ruta_verificacion = (
    MODELS_DIR / "verificacion_modelo_inferencia.npz"
)

TFJS_DIR = MODELS_DIR / "tfjs_model_final"

assert ruta_pesos_npz.exists(), (
    f"No se encontró: {ruta_pesos_npz}"
)

assert ruta_verificacion.exists(), (
    f"No se encontró: {ruta_verificacion}"
)

print("Archivos de entrada encontrados correctamente.")

Archivos de entrada encontrados correctamente.


In [ ]:
def crear_modelo_legacy_inferencia():
    modelo = keras.Sequential(
        [
            keras.layers.Input(
                shape=(64, 64, 1),
                name="Entrada"
            ),

            keras.layers.Conv2D(
                filters=32,
                kernel_size=(3, 3),
                padding="valid",
                activation="relu",
                name="Convolucion_1"
            ),

            keras.layers.MaxPooling2D(
                pool_size=(2, 2),
                name="Pooling_1"
            ),

            keras.layers.Conv2D(
                filters=64,
                kernel_size=(3, 3),
                padding="valid",
                activation="relu",
                name="Convolucion_2"
            ),

            keras.layers.MaxPooling2D(
                pool_size=(2, 2),
                name="Pooling_2"
            ),

            keras.layers.Flatten(
                name="Aplanado"
            ),

            keras.layers.Dense(
                units=64,
                activation="relu",
                name="Densa_64"
            ),

            keras.layers.Dense(
                units=10,
                activation="softmax",
                name="Salida"
            )
        ],
        name="Modelo_Final_Inferencia"
    )

    return modelo


modelo_conversion = crear_modelo_legacy_inferencia()
modelo_conversion.summary()

Model: "Modelo_Final_Inferencia"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Convolucion_1 (Conv2D)      (None, 62, 62, 32)        320       
                                                                 
 Pooling_1 (MaxPooling2D)    (None, 31, 31, 32)        0         
                                                                 
 Convolucion_2 (Conv2D)      (None, 29, 29, 64)        18496     
                                                                 
 Pooling_2 (MaxPooling2D)    (None, 14, 14, 64)        0         
                                                                 
 Aplanado (Flatten)          (None, 12544)             0         
                                                                 
 Densa_64 (Dense)            (None, 64)                802880    
                                                                 
 Salida (Dense)              (None, 10)    

In [ ]:
archivo_pesos = np.load(
    ruta_pesos_npz
)

pesos = [
    archivo_pesos[f"arr_{indice}"]
    for indice in range(len(archivo_pesos.files))
]

modelo_conversion.set_weights(
    pesos
)

print("Cantidad de arreglos de pesos:", len(pesos))
print("Pesos cargados correctamente.")

Cantidad de arreglos de pesos: 8
Pesos cargados correctamente.


In [ ]:
datos_verificacion = np.load(
    ruta_verificacion
)

x_verificacion = datos_verificacion["x"]
salida_original = datos_verificacion["y"]

salida_nueva = modelo_conversion(
    x_verificacion,
    training=False
).numpy()

diferencia_maxima = np.max(
    np.abs(
        salida_original
        - salida_nueva
    )
)

clases_originales = np.argmax(
    salida_original,
    axis=1
)

clases_nuevas = np.argmax(
    salida_nueva,
    axis=1
)

print(
    "Diferencia máxima:",
    f"{diferencia_maxima:.10f}"
)

print(
    "¿Mismas clases?",
    np.array_equal(
        clases_originales,
        clases_nuevas
    )
)

Diferencia máxima: 0.0000000000
¿Mismas clases? True


In [ ]:
ruta_h5_legacy = (
    MODELS_DIR / "modelo_final_inferencia_legacy.h5"
)

modelo_conversion.save(
    ruta_h5_legacy,
    include_optimizer=False
)

print("Modelo HDF5 guardado en:")
print(ruta_h5_legacy)

print(
    "¿Existe?",
    ruta_h5_legacy.exists()
)

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Modelo HDF5 guardado en:
/content/drive/MyDrive/Taller3_Sign_Language/models/modelo_final_inferencia_legacy.h5
¿Existe? True


In [ ]:
if TFJS_DIR.exists():
    shutil.rmtree(TFJS_DIR)

TFJS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Carpeta preparada:")
print(TFJS_DIR)

Carpeta preparada:
/content/drive/MyDrive/Taller3_Sign_Language/models/tfjs_model_final


In [ ]:
tfjs.converters.save_keras_model(
    modelo_conversion,
    str(TFJS_DIR)
)

print("Conversión terminada.")

Conversión terminada.


In [ ]:
etiquetas = list(range(10))

ruta_labels = TFJS_DIR / "labels.json"

with open(
    ruta_labels,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        etiquetas,
        archivo,
        indent=2,
        ensure_ascii=False
    )

print("Etiquetas guardadas.")

Etiquetas guardadas.


In [ ]:
ruta_model_json = TFJS_DIR / "model.json"
archivos_bin = list(
    TFJS_DIR.glob("*.bin")
)

assert ruta_model_json.exists(), (
    "No se generó model.json."
)

assert len(archivos_bin) > 0, (
    "No se generaron archivos .bin."
)

assert ruta_labels.exists(), (
    "No se generó labels.json."
)

print("ARCHIVOS CREADOS")
print("-" * 55)

for archivo in sorted(TFJS_DIR.iterdir()):
    tamaño_kb = archivo.stat().st_size / 1024

    print(
        f"{archivo.name:<35}"
        f"{tamaño_kb:>12.2f} KB"
    )

print()
print("Conversión verificada correctamente.")

ARCHIVOS CREADOS
-------------------------------------------------------
group1-shard1of1.bin                    3212.29 KB
labels.json                                0.05 KB
model.json                                 3.92 KB

Conversión verificada correctamente.
